# METAR Lonboard Visualization
---

In [18]:
import pandas as pd
from datetime import datetime, timedelta, timezone
import geopandas as gpd
from lonboard import viz, Map, ScatterplotLayer, HeatmapLayer
import duckdb
#
import numpy as np
import hvplot.pandas  # noqa
import pgeocode
import holoviews as hv

In [52]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from ipywidgets import FloatRangeSlider, jsdlink
from palettable.colorbrewer.diverging import BrBG_10
from sidecar import Sidecar

from lonboard import Map, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap
from lonboard.controls import MultiRangeSlider
from lonboard.layer_extension import DataFilterExtension

In [53]:
url = 'https://data.source.coop/dynamical/asos-parquet/year=2026/data.parquet'

In [54]:
time_1 = datetime.now()
time_1

datetime.datetime(2026, 6, 16, 21, 11, 28, 594334)

In [66]:
time_0 = datetime.now() - timedelta(hours=72)
time_0

datetime.datetime(2026, 6, 13, 21, 12, 39, 117704)

In [67]:
df = duckdb.execute("""
    SELECT *
    FROM read_parquet($1, hive_partitioning=true)
    WHERE 
---      country = 'US' AND
    valid BETWEEN $2 AND $3
    ORDER BY country
""", [url, time_0, time_1]).fetchdf()

In [68]:
df.head(2)

,station,valid,longitude,latitude,tmpf,tmpc,dwpf,dwpc,relh,drct,...,state,geometry,name,elevation,country,county,wfo,tzname,bbox,year
0,NZQN,2026-06-13 21:30:00+00:00,168.7392,-45.0211,39.2,4.0,32.0,0.0,75.14,30.0,...,AU,"[1, 1, 0, 0, 0, 129, 38, 194, 134, 167, 23, 10...",Queenstown,356.0,AU,None,None,Pacific/Auckland,"{'xmin': 168.7392, 'ymin': -45.0211, 'xmax': 1...",2026
1,NZQN,2026-06-13 22:00:00+00:00,168.7392,-45.0211,41.0,5.0,33.8,1.0,75.32,40.0,...,AU,"[1, 1, 0, 0, 0, 129, 38, 194, 134, 167, 23, 10...",Queenstown,356.0,AU,None,None,Pacific/Auckland,"{'xmin': 168.7392, 'ymin': -45.0211, 'xmax': 1...",2026


In [69]:
df.columns

Index(['station', 'valid', 'longitude', 'latitude', 'tmpf', 'tmpc', 'dwpf',
       'dwpc', 'relh', 'drct', 'sknt', 'gust', 'alti', 'mslp', 'vsby', 'p01i',
       'p01m', 'state', 'geometry', 'name', 'elevation', 'country', 'county',
       'wfo', 'tzname', 'bbox', 'year'],
      dtype='object')

In [70]:
# df[df.where(df.station == "KBDU")]
df_select = df.loc[df['station'] == "BDU"] # to select by station

In [71]:
df_select.head(4)

,station,valid,longitude,latitude,tmpf,tmpc,dwpf,dwpc,relh,drct,...,state,geometry,name,elevation,country,county,wfo,tzname,bbox,year
270864,BDU,2026-06-13 21:15:00+00:00,-105.2258,40.0394,77.0,25.0,35.6,2.0,22.31,0.0,...,CO,"[1, 1, 0, 0, 0, 245, 219, 215, 129, 115, 78, 9...",Boulder,1612.0,US,Boulder,BOU,America/Denver,"{'xmin': -105.2258, 'ymin': 40.0394, 'xmax': -...",2026
270865,BDU,2026-06-13 21:35:00+00:00,-105.2258,40.0394,77.0,25.0,35.6,2.0,22.31,90.0,...,CO,"[1, 1, 0, 0, 0, 245, 219, 215, 129, 115, 78, 9...",Boulder,1612.0,US,Boulder,BOU,America/Denver,"{'xmin': -105.2258, 'ymin': 40.0394, 'xmax': -...",2026
270866,BDU,2026-06-13 21:55:00+00:00,-105.2258,40.0394,78.8,26.0,35.6,2.0,21.02,40.0,...,CO,"[1, 1, 0, 0, 0, 245, 219, 215, 129, 115, 78, 9...",Boulder,1612.0,US,Boulder,BOU,America/Denver,"{'xmin': -105.2258, 'ymin': 40.0394, 'xmax': -...",2026
270867,BDU,2026-06-13 22:15:00+00:00,-105.2258,40.0394,78.8,26.0,35.6,2.0,21.02,120.0,...,CO,"[1, 1, 0, 0, 0, 245, 219, 215, 129, 115, 78, 9...",Boulder,1612.0,US,Boulder,BOU,America/Denver,"{'xmin': -105.2258, 'ymin': 40.0394, 'xmax': -...",2026


In [72]:
variables = {
    'tmpf': "Temperature in Fahrenheit",
    'tmpc': "Temperature in Celsius",
    'dwpf': "Dewpoint in Fahrenheit",
    'dwpc': "Dewpoint in Celsius", 
    'relh': "Relative humidity in percent", 
    'drct': "Wind direction in degrees", 
    'sknt': "Wind speed in knots", 
    'gust': "Wind gusts in knots", 
    'alti': "Altimeter setting in inches of mercury", 
    'mslp': "Mean sea level pressure", 
    'vsby': "Visibility in statute miles", 
    'p01i': "P01I", 
    'p01m': "P01M"
}
print(variables)

{'tmpf': 'Temperature in Fahrenheit', 'tmpc': 'Temperature in Celsius', 'dwpf': 'Dewpoint in Fahrenheit', 'dwpc': 'Dewpoint in Celsius', 'relh': 'Relative humidity in percent', 'drct': 'Wind direction in degrees', 'sknt': 'Wind speed in knots', 'gust': 'Wind gusts in knots', 'alti': 'Altimeter setting in inches of mercury', 'mslp': 'Mean sea level pressure', 'vsby': 'Visibility in statute miles', 'p01i': 'P01I', 'p01m': 'P01M'}


In [73]:
df_foo = pd.DataFrame({
    'date': df_select['valid'],
    'tmpf': df_select['tmpf'],
    'dwpf': df_select['dwpf'],
    'relh': df_select['relh'],
    'drct': df_select['drct'],
    'sknt': df_select['sknt'],
    'alti': df_select['alti'],
}).set_index('date')

In [74]:
df_foo

,tmpf,dwpf,relh,drct,sknt,alti
date,,,,,,
2026-06-13 21:15:00+00:00,77.0,35.6,22.31,0.0,0.0,30.13
2026-06-13 21:35:00+00:00,77.0,35.6,22.31,90.0,5.0,30.14
2026-06-13 21:55:00+00:00,78.8,35.6,21.02,40.0,3.0,30.14
2026-06-13 22:15:00+00:00,78.8,35.6,21.02,120.0,4.0,30.14
2026-06-13 22:35:00+00:00,77.0,35.6,22.31,50.0,4.0,30.14
...,...,...,...,...,...,...
2026-06-16 19:15:00+00:00,87.8,33.8,14.64,120.0,6.0,29.91
2026-06-16 19:35:00+00:00,87.8,33.8,14.64,30.0,5.0,29.89
2026-06-16 19:55:00+00:00,87.8,33.8,14.64,10.0,11.0,29.88


In [76]:
hv.Layout(
    [df_foo[i].hvplot.line(title=f"{variables[i]}", width=400) for i in ['tmpf', 'dwpf', 'relh', 'drct', 'sknt', 'alti']]
).cols(3)

:Layout
   .Curve.Tmpf :Curve   [date]   (tmpf)
   .Curve.Dwpf :Curve   [date]   (dwpf)
   .Curve.Relh :Curve   [date]   (relh)
   .Curve.Drct :Curve   [date]   (drct)
   .Curve.Sknt :Curve   [date]   (sknt)
   .Curve.Alti :Curve   [date]   (alti)

In [27]:
df_select[variables.keys()].head(5)

,tmpf,tmpc,dwpf,dwpc,relh,drct,sknt,gust,alti,mslp,vsby,p01i,p01m
90742,77.0,25.0,37.4,3.0,23.95,180.0,3.0,NaN,30.12,NaN,10.0,0.0,0.0
90743,78.8,26.0,37.4,3.0,22.57,130.0,5.0,NaN,30.11,NaN,10.0,0.0,0.0
90744,78.8,26.0,37.4,3.0,22.57,120.0,5.0,NaN,30.10,NaN,10.0,0.0,0.0
90745,78.8,26.0,37.4,3.0,22.57,160.0,6.0,NaN,30.09,NaN,10.0,0.0,0.0
90746,78.8,26.0,37.4,3.0,22.57,0.0,0.0,NaN,30.09,NaN,10.0,0.0,0.0


In [28]:
data_variable = 'relh'
dates = df_select['valid']
values = df_select[data_variable]
df = pd.DataFrame({'date': dates, 'value': values}).set_index('date')
df.hvplot.line(title=f"Time-Series Line Plot {data_variable}")

:Curve   [date]   (value)

# Plotting DuckDB with lonboard

In [36]:
url = "https://ookla-open-data.s3.us-west-2.amazonaws.com/parquet/performance/type=mobile/year=2019/quarter=1/2019-01-01_performance_mobile_tiles.parquet"
local_path = Path("data-filter-extension.parquet")
if local_path.exists():
    gdf = gpd.read_parquet(local_path)
else:
    columns = ["avg_d_kbps", "avg_u_kbps", "avg_lat_ms", "devices", "tile"]
    df = pd.read_parquet(url, columns=columns)

    tile_geometries = shapely.from_wkt(df["tile"])
    tile_centroids = shapely.centroid(tile_geometries)
    non_geom_columns = [col for col in columns if col != "tile"]
    gdf = gpd.GeoDataFrame(
        df[non_geom_columns],
        geometry=tile_centroids,
        crs="EPSG:4326",
    )
    gdf.to_parquet(local_path)

In [37]:
gdf

,avg_d_kbps,avg_u_kbps,avg_lat_ms,devices,geometry
0,5983,7886,68,1,POINT (-160.01862 70.63722)
1,3748,5841,78,2,POINT (-160.04059 70.63357)
2,3364,6200,78,2,POINT (-160.04059 70.63175)
3,2381,2328,86,1,POINT (-160.0351 70.63357)
4,3047,5356,75,1,POINT (-160.0351 70.63175)
...,...,...,...,...,...
3231240,19528,3200,68,1,POINT (169.81842 -46.29571)
3231241,15693,10359,56,1,POINT (169.81293 -46.3071)
3231242,26747,9674,58,2,POINT (169.66461 -46.42082)
3231243,67995,13564,63,1,POINT (169.65912 -46.4511)


In [39]:
filter_extension = DataFilterExtension(filter_size=3)

In [40]:
min_bound = 5000
max_bound = 50000
normalized_download_speed = (gdf["avg_d_kbps"] - min_bound) / (max_bound - min_bound)
fill_color = apply_continuous_cmap(normalized_download_speed, BrBG_10)
radius = normalized_download_speed * 1000

In [41]:
filter_values = np.column_stack(
    [gdf["avg_d_kbps"], gdf["avg_u_kbps"], gdf["avg_lat_ms"]],
)
initial_filter_range = [
    [10_000, 50_000],
    [1000, 10_000],
    [0, 100],
]

In [42]:
sidecar = Sidecar(anchor="split-right")

In [43]:
layer = ScatterplotLayer.from_geopandas(
    gdf,
    extensions=[filter_extension],
    get_fill_color=fill_color,
    get_radius=radius,
    get_filter_value=filter_values,
    filter_range=initial_filter_range,
    radius_units="meters",
    radius_min_pixels=0.1,
)
m = Map(layer)
download_slider = FloatRangeSlider(
    value=initial_filter_range[0],
    min=0,
    max=70_000,
    step=0.1,
    description="Download: ",
)
upload_slider = FloatRangeSlider(
    value=initial_filter_range[1],
    min=0,
    max=50_000,
    step=1,
    description="Upload: ",
)
latency_slider = FloatRangeSlider(
    value=initial_filter_range[2],
    min=0,
    max=500,
    step=1,
    description="Latency: ",
)
multi_slider = MultiRangeSlider([download_slider, upload_slider, latency_slider])

with sidecar:
    display(m)

In [44]:
# Important: call jsdlink to link the widgets
jsdlink((multi_slider, "value"), (layer, "filter_range"))

multi_slider

MultiRangeSlider(children=(FloatRangeSlider(value=(10000.0, 50000.0), description='Download: ', max=70000.0), …